In [1]:
import pandas as pd
import numpy as np

In [2]:
train_probs_df = pd.read_csv('train_probabilities.csv')
test_probs_df = pd.read_csv('test_probabilities.csv')

In [3]:
train_probs_df.head(3)

,0
0,0.000170
1,0.000005
2,0.001406


In [4]:
train_probs = train_probs_df['0'].to_numpy()
test_probs = test_probs_df['0'].to_numpy()

In [5]:
train_probs

array([1.70220986e-04, 5.49840800e-06, 1.40608029e-03, ...,
       5.79523819e-01, 9.85969381e-01, 5.45847082e-01], shape=(68390,))

In [7]:
# Define number of bins (e.g., 10 for deciles)
num_bins = 10 # Specify number of bins for dividing probabilities into deciles

np.linspace(0, 100, num_bins + 1)

array([  0.,  10.,  20.,  30.,  40.,  50.,  60.,  70.,  80.,  90., 100.])

In [9]:
bin_edges = np.percentile(train_probs, np.linspace(0, 100, num_bins + 1))  # Calculate bin edges based on percentiles
bin_edges

array([8.46086278e-10, 4.02381494e-06, 8.74129940e-05, 1.82997462e-03,
       5.32545916e-02, 6.03632299e-01, 8.92262699e-01, 9.67170529e-01,
       9.90929450e-01, 9.98718522e-01, 9.99999998e-01])

In [10]:
train_bins = np.digitize(train_probs, bin_edges, right=True) - 1  # Bin the train probabilities
test_bins = np.digitize(test_probs, bin_edges, right=True) - 1  # Bin

In [11]:
train_bins

array([2, 1, 2, ..., 4, 7, 4], shape=(68390,))

In [12]:
np.unique(train_bins, return_counts=True)

(array([-1,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9]),
 array([   1, 6838, 6839, 6839, 6839, 6839, 6839, 6839, 6839, 6839, 6839]))

In [13]:
train_bins = np.clip(train_bins, 0, num_bins - 1)  # Ensure bins are within the range [0, num_bins-1]
test_bins = np.clip(test_bins, 0, num_bins - 1)  # Ensure bins are within the range [0, num_bins-1]

np.unique(train_bins, return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([6839, 6839, 6839, 6839, 6839, 6839, 6839, 6839, 6839, 6839]))

In [14]:
train_counts = pd.Series(train_bins).value_counts().sort_index()
test_counts = pd.Series(test_bins).value_counts().sort_index()

train_counts, test_counts

(0    6839
 1    6839
 2    6839
 3    6839
 4    6839
 5    6839
 6    6839
 7    6839
 8    6839
 9    6839
 Name: count, dtype: int64,
 0    2223
 1    2337
 2    2326
 3    2243
 4    1739
 5     574
 6     348
 7     247
 8     218
 9     242
 Name: count, dtype: int64)

In [15]:
train_percents = train_counts * 100 / len(train_probs)
test_percents = test_counts  * 100 / len(test_probs)
train_percents, test_percents

(0    10.0
 1    10.0
 2    10.0
 3    10.0
 4    10.0
 5    10.0
 6    10.0
 7    10.0
 8    10.0
 9    10.0
 Name: count, dtype: float64,
 0    17.788269
 1    18.700488
 2    18.612467
 3    17.948308
 4    13.915340
 5     4.593102
 6     2.784668
 7     1.976474
 8     1.744419
 9     1.936465
 Name: count, dtype: float64)

In [17]:
df = pd.DataFrame({
    'prob_range': [f"{bin_edges[i]:.6f}-{bin_edges[i+1]:.6f}" for i in range(num_bins)],
    'train_count': train_counts.values,
    'train_%': train_percents.values,
    'test_count': test_counts.values,
    'test_%': test_percents.values
})
df

,prob_range,train_count,train_%,test_count,test_%
0,0.000000-0.000004,6839,10.0,2223,17.788269
1,0.000004-0.000087,6839,10.0,2337,18.700488
2,0.000087-0.001830,6839,10.0,2326,18.612467
3,0.001830-0.053255,6839,10.0,2243,17.948308
4,0.053255-0.603632,6839,10.0,1739,13.915340
5,0.603632-0.892263,6839,10.0,574,4.593102
6,0.892263-0.967171,6839,10.0,348,2.784668
7,0.967171-0.990929,6839,10.0,247,1.976474
8,0.990929-0.998719,6839,10.0,218,1.744419
9,0.998719-1.000000,6839,10.0,242,1.936465


In [20]:
# Calculate additional columns
df['A-B'] = df['train_%'] - df['test_%']
df['ln(A/B)'] = np.log((df['train_%'] + 1e-10) / (df['test_%'] + 1e-10))  # Add a small value to avoid log(0)
df['PSI'] = df['A-B']/100 * df['ln(A/B)']
# Replace inf and NAN with 0 in PSI column
df['PSI'] = df['PSI'].replace([np.inf, -np.inf], 0).fillna(0)
df

,prob_range,train_count,train_%,test_count,test_%,A-B,ln(A/B),PSI
0,0.000000-0.000004,6839,10.0,2223,17.788269,-7.788269,-0.575954,0.044857
1,0.000004-0.000087,6839,10.0,2337,18.700488,-8.700488,-0.625965,0.054462
2,0.000087-0.001830,6839,10.0,2326,18.612467,-8.612467,-0.621247,0.053505
3,0.001830-0.053255,6839,10.0,2243,17.948308,-7.948308,-0.584911,0.046491
4,0.053255-0.603632,6839,10.0,1739,13.915340,-3.915340,-0.330407,0.012937
5,0.603632-0.892263,6839,10.0,574,4.593102,5.406898,0.778029,0.042067
6,0.892263-0.967171,6839,10.0,348,2.784668,7.215332,1.278456,0.092245
7,0.967171-0.990929,6839,10.0,247,1.976474,8.023526,1.621270,0.130083
8,0.990929-0.998719,6839,10.0,218,1.744419,8.255581,1.746164,0.144156
9,0.998719-1.000000,6839,10.0,242,1.936465,8.063535,1.641721,0.132381


In [21]:
# Calculate total PSI
total_psi = df['PSI'].sum()
total_psi

np.float64(0.7531824228420665)

### CSI

In [22]:
X_train_1 = pd.read_csv('train_data.csv')
X_test = pd.read_csv('test_data.csv')

In [23]:
# lets understand for one feature

# Bin the features using quantile-based binning
train_binned = pd.qcut(X_train_1['age'], q=4, duplicates='drop')
test_binned = pd.qcut(X_test['age'], q=4, duplicates='drop')

In [24]:
train_binned.head()

0      (0.538, 1.0]
1    (0.404, 0.538]
2    (0.288, 0.404]
3    (0.404, 0.538]
4      (0.538, 1.0]
Name: age, dtype: category
Categories (4, interval[float64, right]): [(-0.001, 0.288] < (0.288, 0.404] < (0.404, 0.538] < (0.538, 1.0]]

In [25]:
test_binned.head()

0     (0.288, 0.423]
1     (0.423, 0.538]
2    (-0.001, 0.288]
3     (0.288, 0.423]
4       (0.538, 1.0]
Name: age, dtype: category
Categories (4, interval[float64, right]): [(-0.001, 0.288] < (0.288, 0.423] < (0.423, 0.538] < (0.538, 1.0]]

In [26]:
# Calculate proportions for each bin
train_proportions = train_binned.value_counts(normalize=True, sort=False)
test_proportions = test_binned.value_counts(normalize=True, sort=False)

In [27]:
train_proportions.head()

age
(-0.001, 0.288]    0.275582
(0.288, 0.404]     0.224899
(0.404, 0.538]     0.258376
(0.538, 1.0]       0.241144
Name: proportion, dtype: float64

In [28]:
test_proportions.head()

age
(-0.001, 0.288]    0.273346
(0.288, 0.423]     0.263903
(0.423, 0.538]     0.216132
(0.538, 1.0]       0.246619
Name: proportion, dtype: float64

In [29]:
# Ensure all bins are represented
all_bins = train_proportions.index.union(test_proportions.index)
train_proportions = train_proportions.reindex(all_bins, fill_value=0)
test_proportions = test_proportions.reindex(all_bins, fill_value=0)

In [30]:
# CSI formula is same as PSI

for bin in all_bins:
    train_pct = train_proportions[bin]
    display(train_pct)

np.float64(0.2755815194195476)

np.float64(0.22489863422962014)

np.float64(0.2583760136577038)

np.float64(0.24114383269312847)

np.float64(0.0)

np.float64(0.0)

In [31]:
for bin in all_bins:
  train_pct = train_proportions[bin]
  test_pct = test_proportions[bin]
  display(test_pct)

np.float64(0.2733456029447067)

np.float64(0.0)

np.float64(0.0)

np.float64(0.24661918860526527)

np.float64(0.2639033368008322)

np.float64(0.2161318716491958)

In [32]:
for bin in all_bins:
  train_pct = train_proportions[bin]
  test_pct = test_proportions[bin]
  A_B = train_pct - test_pct # A minus B
  display(A_B)

np.float64(0.0022359164748408933)

np.float64(0.22489863422962014)

np.float64(0.2583760136577038)

np.float64(-0.005475355912136798)

np.float64(-0.2639033368008322)

np.float64(-0.2161318716491958)

In [33]:
for bin in all_bins:
    train_pct = train_proportions[bin]
    test_pct = test_proportions[bin]
    A_B = train_pct - test_pct
    ln_A_B = np.log(train_pct / test_pct) if test_pct > 0 and train_pct > 0 else 0
    display(ln_A_B)

np.float64(0.008146541679191847)

0

0

np.float64(-0.02245182875563214)

0

0

In [34]:
for bin in all_bins:
    train_pct = train_proportions[bin]
    test_pct = test_proportions[bin]
    A_B = train_pct - test_pct
    ln_A_B = np.log(train_pct / test_pct) if test_pct > 0 and train_pct > 0 else 0
    csi = A_B * ln_A_B
    display(csi) # below csi values are for each bin of age feature

np.float64(1.8214986753483047e-05)

np.float64(0.0)

np.float64(0.0)

np.float64(0.0001229317533154334)

np.float64(-0.0)

np.float64(-0.0)

In [35]:
# Define numerical and categorical features
numerical_features = ['age', 'loan_tenure_months', 'number_of_open_accounts', 'credit_utilization_ratio', 'loan_to_income', 'delinquency_ratio', 'avg_dpd_per_delinquency']
categorical_features = ['residence_type', 'loan_purpose', 'loan_type']

In [36]:
# Initialize lists to store CSI results
numerical_results = []
categorical_results = []

In [37]:
# Calculate CSI for numerical features
for feature in numerical_features:
    # Bin the features using quantile-based binning
    train_binned = pd.qcut(X_train_1[feature], q=4, duplicates='drop')
    test_binned = pd.qcut(X_test[feature], q=4, duplicates='drop')

    # Calculate proportions for each bin
    train_proportions = train_binned.value_counts(normalize=True, sort=False)
    test_proportions = test_binned.value_counts(normalize=True, sort=False)

    # Ensure all bins are represented
    all_bins = train_proportions.index.union(test_proportions.index)
    train_proportions = train_proportions.reindex(all_bins, fill_value=0)
    test_proportions = test_proportions.reindex(all_bins, fill_value=0)

    # Calculate CSI for each bin
    for bin in all_bins:
        train_pct = train_proportions[bin]
        test_pct = test_proportions[bin]
        A_B = train_pct - test_pct
        ln_A_B = np.log(train_pct / test_pct) if test_pct > 0 and train_pct > 0 else 0
        csi = A_B * ln_A_B

        numerical_results.append({
            'Feature': feature,
            'Category/ Bin': bin,
            'Train Count': train_binned[train_binned == bin].count(),
            'Train %': train_pct * 100,
            'Test Count': test_binned[test_binned == bin].count(),
            'Test %': test_pct * 100,
            'A-B': A_B,
            'ln(A/B)': ln_A_B,
            'CSI': csi
        })

In [38]:
# Calculate CSI for categorical features
for feature in categorical_features:
    # Calculate proportions for each category
    train_proportions = X_train_1[feature].value_counts(normalize=True)
    test_proportions = X_test[feature].value_counts(normalize=True)

    # Ensure all categories are represented
    all_categories = train_proportions.index.union(test_proportions.index)
    train_proportions = train_proportions.reindex(all_categories, fill_value=0)
    test_proportions = test_proportions.reindex(all_categories, fill_value=0)

    # Calculate CSI for each category
    for category in all_categories:
        train_pct = train_proportions[category]
        test_pct = test_proportions[category]
        A_B = train_pct - test_pct
        ln_A_B = np.log(train_pct / test_pct) if test_pct > 0 and train_pct > 0 else 0
        csi = A_B * ln_A_B

        categorical_results.append({
            'Feature': feature,
            'Category/ Bin': category,
            'Train Count': X_train_1[feature].value_counts().get(category, 0),
            'Train %': train_proportions[category] * 100,
            'Test Count': X_test[feature].value_counts().get(category, 0),
            'Test %': test_proportions[category] * 100,
            'A-B': A_B,
            'ln(A/B)': ln_A_B,
            'CSI': csi
        })

In [39]:
numerical_df = pd.DataFrame(numerical_results)
categorical_df = pd.DataFrame(categorical_results)
csi_combined_df = pd.concat([numerical_df, categorical_df], ignore_index=True)

In [40]:
# Display combined CSI results
print("Combined CSI Table:")
csi_combined_df['CSI'] = csi_combined_df['CSI'].round(6)
csi_combined_df

Combined CSI Table:


,Feature,Category/ Bin,Train Count,Train %,Test Count,Test %,A-B,ln(A/B),CSI
0,age,"(-0.001, 0.288]",10331,27.558152,3416,27.334560,0.002236,0.008147,0.000018
1,age,"(0.288, 0.404]",8431,22.489863,0,0.000000,0.224899,0.000000,0.000000
2,age,"(0.404, 0.538]",9686,25.837601,0,0.000000,0.258376,0.000000,0.000000
3,age,"(0.538, 1.0]",9040,24.114383,3082,24.661919,-0.005475,-0.022452,0.000123
4,age,"(0.288, 0.423]",0,0.000000,3298,26.390334,-0.263903,0.000000,-0.000000
5,age,"(0.423, 0.538]",0,0.000000,2701,21.613187,-0.216132,0.000000,-0.000000
6,loan_tenure_months,"(-0.001, 0.189]",10152,27.080666,3385,27.086501,-0.000058,-0.000215,0.000000
7,loan_tenure_months,"(0.189, 0.34]",8914,23.778276,2981,23.853725,-0.000754,-0.003168,0.000002
8,loan_tenure_months,"(0.34, 0.547]",9654,25.752241,3238,25.910218,-0.001580,-0.006116,0.000010
9,loan_tenure_months,"(0.547, 1.0]",8768,23.388818,2893,23.149556,0.002393,0.010282,0.000025


In [41]:
# Group by 'Feature' and aggregate by summing up relevant columns
csi_summary_df = csi_combined_df.groupby('Feature').agg({
    'Train Count': 'sum',
    'Test Count': 'sum',
    'CSI': 'sum'
}).reset_index()

In [42]:
csi_summary_df

,Feature,Train Count,Test Count,CSI
0,age,37488,12497,0.000141
1,avg_dpd_per_delinquency,37488,12497,0.000000
2,credit_utilization_ratio,37488,12497,0.000081
3,delinquency_ratio,37488,12497,0.000000
4,loan_purpose,37488,12497,0.000207
5,loan_tenure_months,37488,12497,0.000037
6,loan_to_income,37488,12497,0.000000
7,loan_type,37488,12497,0.000008
8,number_of_open_accounts,37488,12497,0.000047
9,residence_type,37488,12497,0.000192
